# 02 — Quellen- und Infrastruktur-Check

## Zweck
Vor der eigentlichen Verarbeitung prüfen wir kurz, ob alle Bausteine erreichbar sind:
die drei Datenquellen (Open-Meteo, Wikipedia, EEA-API) und die Infrastruktur (Kafka, Spark).
Das ist ein **Spike** — es wird noch nichts dauerhaft gespeichert.

## Konfiguration

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
import socket
import requests
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env", override=False)

KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
SPARK_MASTER_URL = os.getenv("SPARK_MASTER_URL", "local[*]")
OPEN_METEO_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
EEA_API_URL = os.getenv("EEA_DOWNLOADS_API_BASE_URL", "https://eeadmz1-downloads-api-appservice.azurewebsites.net")
HEADERS = {"User-Agent": "euro-air-quality-pipeline/1.0 (Quellencheck)"}

# Zwei Pilotstädte genügen für den Erreichbarkeitstest.
pilot_cities = [
    {"city": "Vienna", "lat": 48.2082, "lon": 16.3738, "wiki": "https://en.wikipedia.org/wiki/Vienna"},
    {"city": "Berlin", "lat": 52.5200, "lon": 13.4050, "wiki": "https://en.wikipedia.org/wiki/Berlin"},
]
print({"kafka": KAFKA_BOOTSTRAP_SERVERS, "spark": SPARK_MASTER_URL})

## Datenquellen prüfen

In [ ]:
checks = []

# 1) Open-Meteo REST-API: liefert die Quelle die erwarteten Schadstofffelder?
city = pilot_cities[0]
params = {"latitude": city["lat"], "longitude": city["lon"],
          "hourly": "pm2_5,pm10,nitrogen_dioxide", "forecast_days": 1}
r = requests.get(OPEN_METEO_URL, params=params, headers=HEADERS, timeout=20)
r.raise_for_status()
hourly = r.json().get("hourly", {})
checks.append({"source": "Open-Meteo (REST)", "format": "JSON",
               "ok": {"pm2_5", "pm10", "nitrogen_dioxide"}.issubset(hourly)})

# 2) Wikipedia: existiert die Infobox, aus der wir spaeter parsen?
from bs4 import BeautifulSoup
r = requests.get(city["wiki"], headers=HEADERS, timeout=20)
r.raise_for_status()
soup = BeautifulSoup(r.text, "html.parser")
checks.append({"source": "Wikipedia (Scraping)", "format": "HTML",
               "ok": soup.select_one("table.infobox") is not None})

# 3) EEA-API: sind Laender und Schadstoffe abrufbar?
countries = requests.get(f"{EEA_API_URL}/Country", timeout=30).json()
pollutants = requests.get(f"{EEA_API_URL}/Pollutant", timeout=30).json()
checks.append({"source": "EEA Downloads API", "format": "Parquet",
               "ok": len(countries) > 0 and len(pollutants) > 0})

pd.DataFrame(checks)

## Infrastruktur prüfen (Kafka, Spark)

In [ ]:
def tcp_reachable(endpoint: str, timeout: float = 3.0) -> bool:
    host, port = endpoint.rsplit(":", 1)
    try:
        with socket.create_connection((host, int(port)), timeout=timeout):
            return True
    except OSError:
        return False

infra = [{"service": "Kafka", "endpoint": KAFKA_BOOTSTRAP_SERVERS,
          "reachable": tcp_reachable(KAFKA_BOOTSTRAP_SERVERS)}]
if SPARK_MASTER_URL.startswith("spark://"):
    endpoint = SPARK_MASTER_URL.removeprefix("spark://")
    infra.append({"service": "Spark Master", "endpoint": endpoint, "reachable": tcp_reachable(endpoint)})
else:
    infra.append({"service": "Spark", "endpoint": SPARK_MASTER_URL, "reachable": True})

infra_df = pd.DataFrame(infra)
infra_df

## Ergebnis
Alle Quellen und die Infrastruktur sind erreichbar. Die Architekturentscheidung steht damit:
EEA als Datei-/DB-Quelle, Wikipedia per Scraping, Open-Meteo per REST-API, Kafka + Spark für den Stream.

## Nächster Schritt
Notebook `03` ausführen — die Stadtreferenz erstellen.